In [ ]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




In [ ]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, dict):   # 🔥 ADD THIS
        return " ".join(map(str, x.values()))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x

    return str(x)

In [ ]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

In [ ]:
df_feat = df.copy()
df_feat.sample(5)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")

In [ ]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [ ]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [ ]:
def extract_features(text):
    doc = nlp(text)

    tokens = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            "chat_word_count": 0,
            "punct_count": 0,
            "function_count": 0,
            "discourse_count": 0,
            # spelling_error_ratio removed: is_oov flags proper nouns/technical
            # terms as errors, introducing more noise than signal
            "ttr": 0,
            "function_word_ratio": 0,
            "discourse_ratio": 0,
            "sentence_length": 0,
            "avg_word_length": 0,
        }

    chat_word_count = sum(1 for t in tokens if t in chat_words)
    punct_count = sum(1 for t in doc if t.is_punct)

    function_count = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    return {
        "chat_word_count": chat_word_count,
        "punct_count": punct_count,
        "function_count": function_count,
        "discourse_count": discourse_count,
        # spelling_error_ratio removed: spaCy is_oov misclassifies domain
        # terms, proper nouns, and abbreviations as spelling errors,
        # producing near-constant noise (ratio ~1.0 across all samples)
        "ttr": len(set(alpha_tokens)) / total_alpha,
        "function_word_ratio": function_count / total_tokens,
        "discourse_ratio": discourse_count / total_tokens,
        "sentence_length": total_tokens,
        "avg_word_length": sum(len(t) for t in alpha_tokens) / total_alpha,
    }

In [ ]:
def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]

    if len(feats) == 0:
        return {}

    df = pd.DataFrame(feats)

    return {
        "chat_word_count": df["chat_word_count"].sum(),
        "punct_count": df["punct_count"].sum(),

        "function_count": df["function_count"].sum(),
        "discourse_count": df["discourse_count"].sum(),

        # spelling_error_ratio removed — see extract_features for rationale
        "ttr": df["ttr"].mean(),
        "function_word_ratio": df["function_word_ratio"].mean(),
        "discourse_ratio": df["discourse_ratio"].mean(),

        "sentence_length_mean": df["sentence_length"].mean(),
        # fillna(0): single-sentence docs produce NaN std; 0 is the correct
        # value (no variance when there is only one sentence)
        "sentence_length_std": df["sentence_length"].std(ddof=1) if len(df) > 1 else 0.0,

        "avg_word_length": df["avg_word_length"].mean(),
    }

In [ ]:
def build_final_features(df):
    rows = []

    for _, row in df.iterrows():
        text = row["answers"]
        label = row["label"]

        # 1. split into sentences
        sentences = split_sentences(text)

        # 2. aggregate sentence-level features → document-level features
        features = aggregate_sentence_features(sentences)

        # 3. attach label
        features["label"] = label

        rows.append(features)

    return pd.DataFrame(rows)

In [ ]:
df_final = build_final_features(df_feat)

In [ ]:
import joblib
joblib.dump(df_final, "final_features.pkl")

In [ ]:
import joblib
df_feat = joblib.load("final_features.pkl")

In [ ]:
df_feat.head()

In [ ]:
df_final = pd.concat(
    [
        df[["answers", "label"]].reset_index(drop=True),
        df_feat.drop(columns=["label"], errors="ignore").reset_index(drop=True)
    ],
    axis=1
)
df_final.head()

In [ ]:
df_feat.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
X_ans = df_final["answers"]
y = df_final["label"]
X_custom = df_final.drop(columns=["answers", "label"])

In [ ]:
X_ans_train, X_ans_test, X_custom_train, X_custom_test, y_train, y_test = train_test_split(
    X_ans,
    X_custom,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain class balance
)

In [ ]:

import numpy as np

print(X_custom_train.isna().sum())

In [ ]:
X_custom_train = X_custom_train.fillna(0)
X_custom_test= X_custom_test.fillna(0)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_ans_train)
X_test_tfidf = tfidf.transform(X_ans_test)


In [ ]:
feature_cols = X_custom_train.columns.tolist()
joblib.dump(feature_cols, "feature_columns.pkl")
scaler = StandardScaler(with_mean=False) 
X_train_custom_scaled = scaler.fit_transform(X_custom_train)
X_test_custom_scaled = scaler.transform(X_custom_test)

In [ ]:
X_train = hstack([X_train_tfidf, X_train_custom_scaled])
X_test = hstack([X_test_tfidf, X_test_custom_scaled])


In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance if needed
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

# Evaluate model
print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print("Confusion Matrix on Test Set:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
joblib.dump(model, 'logistic_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')


In [ ]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack

# Load saved artifacts
model = joblib.load('logistic_model.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')
scaler = joblib.load('feature_scaler.pkl')
feature_cols = joblib.load('feature_columns.pkl')  # saved column order from training

# Custom feature extraction functions must be defined above:
# split_sentences, clean_text, extract_features, aggregate_sentence_features

# Input text
text = """In this pipeline, I am performing data cleaning before passing features into the model. First, I replace all missing values (NaN) with 0 to ensure there are no undefined values in the dataset. Then, I handle infinite values (positive and negative infinity) by replacing them with 0, since such values can occur due to division operations in feature engineering. Finally, I align the feature DataFrame with the exact column order used during training by reindexing using the saved feature list. This ensures that the model receives input in the same structure it was trained on, which is critical for correct predictions and avoiding shape or alignment errors."""

# Step 1: Split text into sentences
sentences = split_sentences(text)

# Step 2: Create chunks (same granularity used conceptually during training)
chunk_size = 3  # Adjust as needed
chunks = [' '.join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

# Step 3: Extract features for each chunk and aggregate
chunk_features = []
for chunk in chunks:
   sentences_in_chunk = [clean_text(s) for s in split_sentences(chunk)]
   aggregated_features = aggregate_sentence_features(sentences_in_chunk)
   chunk_features.append(aggregated_features)

# Step 4: Convert chunk features to DataFrame and sanitize
features_df = pd.DataFrame(chunk_features)
features_df = features_df.replace([np.inf, -np.inf], 0)  # guard division artefacts
features_df = features_df.fillna(0)                       # guard single-sentence std NaN

# Step 5: Align column order with training (done once — duplicate removed)
features_df = features_df.reindex(columns=feature_cols, fill_value=0)

# Step 6: Scale custom features
custom_scaled = scaler.transform(features_df)

# Step 7: Transform each chunk using TF-IDF
chunk_tfidf = tfidf.transform(chunks)

# Step 8: Combine TF-IDF and custom features
X_chunk = hstack([chunk_tfidf, custom_scaled])

# Step 9: Predict probabilities for each chunk
chunk_probs = model.predict_proba(X_chunk)[:, 1]  # probability of class 1 (AI)

# Step 10: Aggregate chunk probabilities (average)
final_score = np.mean(chunk_probs)

# Step 11: Final prediction using threshold
final_prediction = 1 if final_score > 0.5 else 0

print("Final aggregated score:", final_score)
print("Final prediction:", "AI" if final_prediction == 1 else "Human")
